In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on D20, cooling scan

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the olivine Co₂SiO₄ with its magnetic structure to the whole D20 cooling scan, 213 patterns
from 497.4 K down to 2.3 K, one after another and from the lowest temperature up. First the fit
is staged on the 2.3 K pattern, the scan's template; then each later pattern starts from the
previous result. The cobalt moments stay free at every temperature and fall towards zero near
the Néel temperature, about 50 K.

The project is the catalogue case `pd-neut-cwl_cosio-d20_scan-213f`, the one the command line
and the app's Examples open. Its two Co sites share one displacement from the start.

In [2]:
import csv
from pathlib import Path

import easydiffraction as edi

## Load the project

A sequential fit writes one row per pattern into the project's `analysis/results.csv`, so work
on a copy of the project, saved to a folder of its own.

In [3]:
project_path = edi.download_data('pd-neut-cwl_cosio-d20_scan-213f', destination='data')
project = edi.Project.load(project_path)
project.save_as('pd-neut-cwl_cosio-d20_scan-213f')
structure = project.structures['cosio']
experiment = project.experiments['d20']

Data 'pd-neut-cwl_cosio-d20_scan-213f' is at data/pd-neut-cwl_cosio-d20_scan-213f-a86d87ed4b81/pd-neut-cwl_cosio-d20_scan-213f
Saving project to 'pd-neut-cwl_cosio-d20_scan-213f'


## Stage the template fit

The template is the 2.3 K pattern. Fit it alone first, in stages, so the scan starts from a
good model: switch the fitting mode to a single fit, and start with nothing free.

In [4]:
project.analysis.fitting_mode = 'single'
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('d20')

View(html=<305,983 characters>, height=480)

## Perform the analysis

### Scale, 2θ zero and cell

In [6]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3081,,0,30,Å
2,cosio,cell,,length_b,6.00219,,0,30,Å
3,cosio,cell,,length_c,4.78739,,0,30,Å
4,d20,instrument,,calib_twotheta_offset,0.29433,,-inf,inf,deg
5,d20,linked_structure,cosio,scale,1.81723,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 5 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       15.23        64.02
      1      0.004     14.24        55.94   12.6% ↓
      2      0.007     14.22        55.83   0.2% ↓
      3      0.011     14.22        55.83   
      4      0.016     14.22        55.83   
  done · converged · 4 iters · reduced χ² 55.83 · Rwp 14.22 % · 0.02 s


View(html=<82,748 characters>, height=376)

In [9]:
project.display.pattern('d20')

View(html=<307,327 characters>, height=480)

### Background

The sixteen background points follow the level between the peaks.

In [10]:
for point in experiment.background:
    point.intensity.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.22        56.46
      1      0.005     13.46        50.59   10.4% ↓
      2      0.011     13.46        50.59   
  done · converged · 2 iters · reduced χ² 50.59 · Rwp 13.46 % · 0.01 s


View(html=<98,467 characters>, height=376)

In [12]:
project.display.pattern('d20')

View(html=<307,350 characters>, height=480)

### Peak widths

U, V and W only: across a temperature scan the Lorentzian term and the asymmetry stay at their
2.3 K values, as do the form factor's j2 term.

In [13]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.46        50.70
      1      0.005     13.29        49.39   2.6% ↓
      2      0.009     13.29        49.35   0.1% ↓
      3      0.014     13.28        49.35   
  done · converged · 3 iters · reduced χ² 49.35 · Rwp 13.28 % · 0.02 s


View(html=<101,745 characters>, height=376)

In [15]:
project.display.pattern('d20')

View(html=<307,496 characters>, height=480)

### Atom positions

In [16]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 35 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.28        49.74
      1      0.006     13.22        49.28   0.9% ↓
      2      0.010     13.22        49.27   
      3      0.016     13.22        49.27   
  done · converged · 3 iters · reduced χ² 49.27 · Rwp 13.22 % · 0.02 s


View(html=<113,210 characters>, height=376)

In [18]:
project.display.pattern('d20')

View(html=<307,413 characters>, height=480)

### Displacements

Co2's displacement follows Co1's by the project's constraint.

In [19]:
for site in structure.atom_sites:
    site.adp_iso.free = not site.adp_iso.user_constrained

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 40 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.22        49.45
      1      0.006     13.01        47.86   3.2% ↓
      2      0.011     13.00        47.83   0.1% ↓
      3      0.017     13.00        47.83   
  done · converged · 3 iters · reduced χ² 47.83 · Rwp 13.00 % · 0.02 s


View(html=<118,485 characters>, height=376)

In [21]:
project.display.pattern('d20')

View(html=<307,551 characters>, height=480)

### Magnetic moments

Last, the moments: Co1 in any direction, Co2 along b.

In [22]:
structure.atom_site_moment['Co1'].crystalaxis_x.free = True
structure.atom_site_moment['Co1'].crystalaxis_y.free = True
structure.atom_site_moment['Co1'].crystalaxis_z.free = True
structure.atom_site_moment['Co2'].crystalaxis_y.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 44 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.00        47.97
      1      0.008     10.87        33.54   30.1% ↓
      2      0.012      4.94         6.91   79.4% ↓
      3      0.016      4.71         6.30   8.8% ↓
      4      0.020      4.69         6.25   0.8% ↓
      5      0.028      4.69         6.24   0.2% ↓
      6      0.033      4.69         6.24   
      7      0.038      4.69         6.24   
      8      0.043      4.69         6.24   
      9      0.048      4.69         6.24   
     10      0.053      4.69         6.24   
     11      0.057      4.69         6.24   
     12      0.064      4.69         6.24   
     13      0.068      4.69         6.24   
     14      0.073      4.69         6.24   
     15      0.078      4.69         6.24   
     16      0.083      4.69         6.24   
     17      0.088      4.69         6.24   
     18      0.092    

View(html=<122,717 characters>, height=376)

In [24]:
project.display.pattern('d20')

View(html=<307,360 characters>, height=480)

## Fit the scan

Back to the sequential mode, with the template's free parameters. The fit can stop between
patterns: here it stops after five, all below 3 K, and the rows it wrote stay.

In [25]:
project.analysis.fitting_mode = 'sequential'
fitted = []
result = project.fit_sequential(
    on_file_complete=lambda *args: fitted.append(args),
    should_cancel=lambda: len(fitted) >= 5,
)
print(result.status.name.lower(), 'after', len(fitted), 'patterns')

cancelled after 5 patterns


## Resume

Starting again continues with the first pattern that has no row yet, from the last fitted
values. The whole scan takes a while.

In [26]:
result = project.fit_sequential(on_file_complete=lambda *args: fitted.append(args))
print(result.status.name.lower(), 'with', len(fitted), 'patterns fitted in all')

done with 213 patterns fitted in all


## The moments against temperature

One row per pattern, in the order they were fitted. The moment columns name their site and
component.

In [27]:
results = Path('pd-neut-cwl_cosio-d20_scan-213f/analysis/results.csv')
with results.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
moments = [name for name in rows[0] if 'crystalaxis' in name and not name.endswith('_su')]
print(len(rows), 'rows; moment columns:', ', '.join(moments))

213 rows; moment columns: cosio.atom_site_moment.Co1.crystalaxis_x, cosio.atom_site_moment.Co1.crystalaxis_x.uncertainty, cosio.atom_site_moment.Co1.crystalaxis_y, cosio.atom_site_moment.Co1.crystalaxis_y.uncertainty, cosio.atom_site_moment.Co1.crystalaxis_z, cosio.atom_site_moment.Co1.crystalaxis_z.uncertainty, cosio.atom_site_moment.Co2.crystalaxis_y, cosio.atom_site_moment.Co2.crystalaxis_y.uncertainty


## Save the report

The HTML report draws the measured, calculated and difference patterns with the nuclear and the
magnetic reflection positions, and lists the structure and the fit. Open it in a browser.

In [28]:
project.report.save_html('pd-neut-cwl_cosio-d20_scan-213f.html')

Saving report to 'pd-neut-cwl_cosio-d20_scan-213f.html'
